In [1]:
!pip install -q transformers torch pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 12.6 MB/s eta 0:00:00


In [2]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()

pdf_path = next(iter(uploaded))

reader = PdfReader(pdf_path)

document_text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        document_text += page_text + "\n"

print("PDF uploaded successfully!")
print("Total characters:", len(document_text))
print("\nFirst 1000 characters:\n")
print(document_text[:1000])

Saving ApplicationForm (2).pdf to ApplicationForm (2).pdf
PDF uploaded successfully!
Total characters: 2716

First 1000 characters:

(Enrollment ID)M610U43
Zone/Scrutiny Code
6EW
MEDANDRAO SRINIVAS
(Full Name of the Applicant)
Indian
National
MEDANDRAO VENKATA NAGA JYOTHI
GATE Office
Kharagpur 721302
gateonline@adm.iitkgp.ac.in, 03222-282091
Digital Fingerprint : bfa75294946b3c43349b6866eae40bb9
 
78******29
GATE Paper Code
CS
Indian Institute of Technology Kharagpur
(Parent/Guardian's Mobile Number) (Parent/Guardian's Name)
M610U43
 
(Computer Science and
Information Technology)
e-Signature : Medandrao Srinivas
Payment Mode Transaction Ref. No Transaction Date Amount Payment For
Unified Payments 114881274701 10/5/26, 8:45PM 2000.0 Application Fee
(D.O.B)
27/07/2005 Male GEN-EWS
(Gender) (Category)
Exam City 1 : Vijayawada (AP)
Exam City 2 : Eluru (AP)
Exam City 3 : Bhimavaram (AP)
GATE Exam Details
Zonal GATE Office Contact
Payment Details
Communication Address
14-34
Poranki
Krishna
A

In [3]:
def split_text(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = split_text(document_text)

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0] if chunks else "No text found.")

Total chunks: 7

First chunk:

(Enrollment ID)M610U43
Zone/Scrutiny Code
6EW
MEDANDRAO SRINIVAS
(Full Name of the Applicant)
Indian
National
MEDANDRAO VENKATA NAGA JYOTHI
GATE Office
Kharagpur 721302
gateonline@adm.iitkgp.ac.in, 03222-282091
Digital Fingerprint : bfa75294946b3c43349b6866eae40bb9
 
78******29
GATE Paper Code
CS
Indian Institute of Technology Kharagpur
(Parent/Guardian's Mobile Number) (Parent/Guardian's Name)
M610U43
 
(Computer Science and
Information Technology)
e-Signature : Medandrao Srinivas
Payment Mode


In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Embeddings created successfully!")
print("Number of chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings created successfully!
Number of chunks: 7
Embedding shape: torch.Size([7, 384])


In [7]:
from sentence_transformers import util

def retrieve_relevant_chunks(question, top_k=3):
    question_embedding = embedding_model.encode(
        question,
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        question_embedding,
        chunk_embeddings
    )[0]

    k = min(top_k, len(chunks))
    top_results = torch.topk(similarities, k=k)

    retrieved_chunks = []

    for score, index in zip(
        top_results.values,
        top_results.indices
    ):
        retrieved_chunks.append({
            "text": chunks[index.item()],
            "similarity": score.item()
        })

    return retrieved_chunks

In [9]:
question = "What information is provided about the examination?"

results = retrieve_relevant_chunks(question)

for i, result in enumerate(results, start=1):
    print(f"\nPassage {i}")
    print(f"Similarity: {result['similarity']:.2%}")
    print(result["text"])
    print("-" * 60)


Passage 1
Similarity: 42.40%
ed by me is found to be incorrect later, I understand that my candidature for GATE 2027 may be cancelled, before, during, or
after the examination, including the time after generation of my GATE 2027 Score Card. Further, I understand that I may be liable for legal action for
providing false information. GATE 2027 Examination Body’s decision will be final and legally binding on me.
I also understand that GATE is NOT an admission ensuring examination, and qualifying in GATE examination does NOT gu
------------------------------------------------------------

Passage 2
Similarity: 40.86%
ntact
Payment Details
Communication Address
14-34
Poranki
Krishna
Andhra Pradesh 521137
91-99******44 (Mobile No.)
me*************as@gmail.com
Qualification Details
B. Tech.
Data Science
Graduated: Yes; Year of Graduation: 2026
NRI INSTITUTE OF TECHNOLOGY
Andhra Pradesh 521212
I confirm that I fulfil the eligibility criteria to appear for the Graduate Aptitude Test in Enginee

In [10]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

qa_model_name = "distilbert-base-cased-distilled-squad"

qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)

print("RAG question-answering model loaded successfully!")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

RAG question-answering model loaded successfully!


In [11]:
def answer_pdf_question(question, top_k=3):
    retrieved = retrieve_relevant_chunks(question, top_k=top_k)

    context = "\n".join(
        item["text"] for item in retrieved
    )

    inputs = qa_tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_index = torch.argmax(outputs.start_logits).item()
    end_index = torch.argmax(outputs.end_logits).item()

    if end_index < start_index:
        return "Could not identify an answer.", retrieved

    answer_tokens = inputs["input_ids"][0][start_index:end_index + 1]

    answer = qa_tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    if not answer.strip():
        answer = "Could not identify an answer."

    return answer, retrieved

In [12]:
while True:
    question = input("\nAsk a question about your PDF (or type 'exit'): ")

    if question.lower() == "exit":
        print("PDF Question Answering stopped.")
        break

    answer, sources = answer_pdf_question(question)

    print("\nAnswer:")
    print(answer)

    print("\nRelevant passages:")
    for i, source in enumerate(sources, start=1):
        print(f"\nSource {i} (similarity: {source['similarity']:.2%})")
        print(source["text"])
        print("-" * 50)


Ask a question about your PDF (or type 'exit'): What examination details are mentioned?

Answer:
Could not identify an answer.

Relevant passages:

Source 1 (similarity: 44.25%)
ntact
Payment Details
Communication Address
14-34
Poranki
Krishna
Andhra Pradesh 521137
91-99******44 (Mobile No.)
me*************as@gmail.com
Qualification Details
B. Tech.
Data Science
Graduated: Yes; Year of Graduation: 2026
NRI INSTITUTE OF TECHNOLOGY
Andhra Pradesh 521212
I confirm that I fulfil the eligibility criteria to appear for the Graduate Aptitude Test in Engineering (GATE 2027) examination. I have read the Information
Brochure and filled up the GATE 2027 Application Form myself. I
--------------------------------------------------

Source 2 (similarity: 42.92%)
til the validity of the Score card.
I agree that I am responsible for verifying my eligibility for the GATE examination and for fulfilling the requirements of the qualifying degree.
Application Form
Exam City 4 : Tadepalligudem (AP)
Exam C